# Trabajo Fin de Máster  
### Análisis de la Ciudad mediante Aprendizaje Supervisado  
#### Detección Automática de Tipologías Residenciales y Patrones de Cerramiento: Interpretabilidad vs Rendimiento

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

> **Autor:** David Fernández Martínez    
> **Email personal:** david.fernxndez.martinez@gmail.com  
> **Email académico:** davidfm8@correo.ugr.es  
> **LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://www.linkedin.com/in/david-fern%C3%A1ndez-mart%C3%ADnez/)  
> **GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Generación de las particiones (*folds*) para la validación cruzada anidada (Nested Cross Validation)

### 📝 Descripción del notebook
En este notebook se presenta la estrategia de validación cruzada *Nested Cross Validation* utilizada para evaluar de forma robusta el rendimiento de distintos modelos de clasificación. Se explican las ventajas de esta metodología frente a otras estrategias de validación, así como el procedimiento empleado para garantizar la reproducibilidad de los experimentos mediante la generación y almacenamiento de las particiones (*folds*) en ficheros CSV.

### Indice de contenidos
1. [Estrategia de validación](#nestrategia_validacion)
    * [1.1 Limitaciones de los enfoques tradicionales](#limitaciones)
    * [1.2 Validación cruzada anidada (*Nested Cross-Validation*)](#nested)

2. [Configuración](#implementacion)

3. [Generación de los *folds* (reproducibilidad)](#reproducibilidad)
    * [3.1 Formato de los ficheros CSV](#formato_ficheros_csv)
        * [3.1.1 Fichero para *Outer CV*: *outer_folds.csv*](#outer_folds_csv)
        * [3.1.2 Ficheros para *Inner CV*: *inner_fold_{outer_fold_idx}.csv*](#inner_folds_csv)    

4. [Validación de los folds creados](#validacion)

# Configuración de entorno e *imports*

Este proyecto ha sido realizado en un entorno Anaconda con la versión 3.11.15 de *Python*. Las versiones de las librerias requeridas se encuentran en el fichero *requirements-full.txt*.

En esta sección se importan las librerias necesarias para la ejecución de este fichero *jupyter notebook*, se activa el *reload* de módulos externos y se configuran aspectos globales y de reproducibilidad.

In [2]:
# jupyter extensions to automatically reload external modules
%load_ext autoreload
%autoreload 2

In [1]:
import warnings
import random
import numpy as np
import pandas as pd

# Configuration object
from src.config import cfg

# Nested Cross Validation methods
from src.nested_cv_utils import generate_folds, validate_folds

**Solución a problemas en *imports***

Si los *imports* del módulo `src` fallan al ejecutar este cuaderno en un entorno diferente, descomente y ejecute la siguiente celda:

```python
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
```

In [3]:
# Global configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

In [4]:
# Reproducibility
SEED = cfg.SEED 
np.random.seed(SEED)
random.seed(SEED)

<a id="estrategia_validacion"></a>
# 1. Estrategia de validación

La estrategia de validación constituye uno de los elementos fundamentales del protocolo experimental, ya que determina el procedimiento mediante el cual se estima la capacidad de generalización de los modelos. La elección de una estrategia adecuada resulta especialmente relevante cuando se trabaja con un conjuntos de datos de tamaño reducido, donde una utilización ineficiente de las observaciones puede introducir estimaciones sesgadas del rendimiento. En las siguientes secciones se analizan las limitaciones de los enfoques de validación convencionales en este contexto, se justifica la adopción de la validación cruzada anidada *Nested Cross-Validation* y se describe la configuración empleada para garantizar la reproducibilidad del experimento.

<a id="limitaciones"></a>
## 1.1 Limitaciones de los enfoques tradicionales

Los métodos de validación clásicos (*Hold-Out* y *Cross-Validation*) presentan limitaciones importantes cuando se aplican a conjuntos de datos con un número reducido de muestras.

El enfoque *Hold-Out* divide el conjunto de datos en un conjunto de entrenamiento y otro de prueba mediante una única partición. Como consecuencia, la estimación del rendimiento depende de las muestras asignadas a cada subconjunto. En conjuntos de datos reducidos, esta dependencia se acentúa, ya que una única partición difícilmente representa toda la variabilidad de los datos, incrementando la varianza de la estimación y reduciendo la fiabilidad de los resultados.

La validación cruzada tradicional (*Cross-Validation*) aprovecha de forma eficiente los datos disponibles, ya que cada muestra participa tanto en el entrenamiento como en la evaluación del modelo en distintas iteraciones. No obstante, cuando la misma validación cruzada se utiliza simultáneamente para optimizar el modelo (selección de hiperparámetros) y estimar el rendimiento final, la evaluación resulta optimista, puesto que el modelo ha sido optimizado específicamente sobre esos datos. Este fenómeno, conocido como sesgo de selección de hiperparámetros, conduce a una subestimación del error real de generalización y resulta especialmente acusado en conjuntos de datos reducidos.

Una alternativa consiste en reservar un conjunto de prueba independiente para realizar la evaluación final tras la optimización de los hiperparámetros. Sin embargo, en conjuntos de datos pequeños esta estrategia vuelve a depender de una única partición, por lo que la estimación del rendimiento puede presentar una alta varianza condicionada a las muestras incluidas en dicho conjunto de prueba.

Para aprovechar todos los datos disponibles y, al mismo tiempo, obtener una evaluación no sesgada del rendimiento, se emplea la estrategia de validación cruzada anidada (*Nested Cross-Validation*).

<a id="nested"></a>
## 1.2 Validación cruzada anidada (*Nested Cross-Validation*)

Este enfoque separa completamente el proceso de selección de hiperparámetros de la evaluación del rendimiento del modelo, lo que resulta en una estimación muy cercana a la que se obtendría con un conjunto de prueba independiente. Para ello, se implementan dos bucles de validación cruzada anidados:

* **Validación cruzada externa (*Outer CV Loop*)**. Se utiliza para evaluar el rendimiento del modelo sobre datos que no han intervenido en el proceso de selección de hiperparámetros. Para ello, el conjunto de datos se divide en $K_{out}$ particiones (*folds*).

* **Validación cruzada interna (*Inner CV Loop*)**. Se aplica exclusivamente sobre el conjunto de entrenamiento definido en el bucle externo y tiene como objetivo seleccionar la combinación óptima de hiperparámetros. Para ello, dicho conjunto de entrenamiento se divide en $K_{in}$ particiones (*folds*).

Al finalizar el procedimiento, se obtienen $K_{out}$ estimaciones independientes del rendimiento, una por cada iteración del bucle externo, lo que permite calcular métricas promedio y analizar su variabilidad. De este modo, la evaluación aprovecha la totalidad del conjunto de datos sin introducir el sesgo asociado a utilizar los mismos datos tanto para la selección de hiperparámetros como para la estimación del rendimiento.

En consecuencia, la validación cruzada anidada evalúa el procedimiento completo de aprendizaje, incluyendo la selección de hiperparámetros y el entrenamiento del modelo, en lugar de valorar un modelo ajustado sobre una partición concreta de los datos. Esto proporciona una estimación de la capacidad del algoritmo para aprender a partir de los datos disponibles y generalizar ante nuevas observaciones.


A continuación, se presenta la formulación matemática de la validación cruzada anidada, definiendo formalmente la construcción de las particiones correspondientes a los bucles externo e interno, así como el procedimiento seguido para la estimación del rendimiento.

**Formulación matemática**

En el bucle externo, el conjunto de datos original $D$ se divide en $K_{out}$ particiones disjuntas de tamaño equivalente, cumpliendo:

$$
D = \bigcup_{i=1}^{K_{out}} D_i^{out} \quad \text{donde} \quad D_i^{out} \cap D_j^{out} = \emptyset \text{ para } i \neq j
$$

El número $K_{out}$ representa el número de particiones (*folds*) utilizadas en la validación cruzada externa. En cada iteración $i$, con $i \in \{1,2,\dots,K_{out}\}$, una de las particiones se reserva como conjunto de prueba externo resultando en dos subconjuntos:

* *Outer Test Set*. Corresponde a la partición $D_i^{out}$ y es el conjunto de prueba externo utilizado exclusivamente para la evaluación final del modelo en la iteración $i$.

* *Outer Train Set*. Está formado por el resto de particiones y se utiliza para el entrenamiento y selección de hiperparámetros del modelo. Se define como el complemento del *Outer Test Set*:

$$
D_{-i}^{out} = D \setminus D_i^{out}
$$

Sobre cada conjunto *Outer Train Set* se ejecuta el proceso de selección de hiperparámetros mediante el bucle interno de validación cruzada. Para ello, el conjunto $D_{-i}^{out}$ se divide en $K_{in}$ particiones disjuntas:

$$
D_{-i}^{out} = \bigcup_{j=1}^{K_{in}} D_{ij}^{in} \quad \text{donde} \quad D_{ij}^{in} \cap D_{ik}^{in} = \emptyset \text{ para } j \neq k
$$

El número $K_{in}$ representa el número de particiones utilizadas en la validación cruzada interna. Para cada iteración $j$, con $j\in\{1,2,\dots,K_{in}\}$, se definen:

* *Inner Validation Set*. Corresponde a la partición $D_{ij}^{in}$ y se utiliza para evaluar cada configuración de hiperparámetros.

* *Inner Train Set*. Está compuesto por las restantes particiones del bucle interno y se define como:

$$
D_{i,-j}^{in} = D_{-i}^{out} \setminus D_{ij}^{in}
$$

Para cada configuración de hiperparámetros, el modelo se entrena sobre el conjunto *Inner Train Set* y se evalúa sobre el correspondiente *Inner Validation Set*. La configuración seleccionada es aquella que obtiene el mejor rendimiento promedio sobre las $K_{in}$ particiones internas.

Finalmente, el modelo se entrena utilizando la configuración óptima sobre todo el conjunto *Outer Train Set* ($D_{-i}^{out}$) y se evalúa sobre el conjunto *Outer Test Set* ($D_i^{out}$), que se ha mantenido independiente a la selección de hiperparámetros.

Este procedimiento se repite para las $K_{out}$ particiones del bucle externo, obteniendo $K_{out}$ estimaciones independientes del rendimiento del modelo.


<a id="implementacion"></a>
# 2 Configuración 

La implementación práctica de la validación cruzada anidada requiere definir el número de particiones (*folds*) de los bucles externo e interno. En este trabajo se ha seleccionado una configuración de $K_{out}=5$ y $K_{in}=5$, buscando un equilibrio entre la robustez estadística de la estimación del rendimiento, la estabilidad del proceso de selección de hiperparámetros y el coste computacional asociado al entrenamiento de los modelos.

In [6]:
# Nested Cross-Validation configuration are set in src/config.py
print(f"K_out={cfg.OUTER_SPLITS}")
print(f"K_in={cfg.INNER_SPLITS}")

K_out=5
K_in=5


Con esta configuración, cada iteración del bucle externo reserva un $20%$ de los datos como conjunto de prueba (*Outer Test Set*), mientras que el $80%$ restante se utiliza para la selección de hiperparámetros mediante la validación cruzada interna. La elección de $K_{in}=5$ permite realizar una selección de hiperparámetros robusta manteniendo un número suficiente de muestras de cada clase en las particiones internas. Un número elevado de *folds* podría reducir la representación de las clases minoritarias en un conjunto de datos reducido y desbalanceado. Para preservar la distribución original de las clases en cada subconjunto, las particiones del bucle externo e interno se han generado de forma estratificada utilizando la clase *StratifiedKFold* del módulo *model_selection* de *scikit-learn*.

En la siguiente figura se ilustra de forma gráfica la configuración de la estrategia de validación cruzada anidada, mostrando la relación entre los bucles externo e interno.

<img src="../images/experiment/Nested_cross_validation.png" alt="Nested Cross-Validation" width="90%">

<a id="reproducibilidad"></a>
# 3. Generación de los *folds* (reproducibilidad)

Para garantizar la reproducibilidad del protocolo experimental de evaluación de rendimiento, las particiones utilizadas en la validación cruzada anidada se generan utilizando una semilla fija definida en el fichero de configuración del proyecto y se almacenan en ficheros *CSV* dentro del directorio *data/folds*. El proceso de generación se encuentra encapsulado en la función *generate_folds()*, ubicada en el módulo *src/nested_cv_utils.py*.

A continuación se muestra la ejecución de dicha función. Para generar *folds* alternativos deben configurarse las variables `OUTER_SPLITS` e `INNER_SPLITS` en *src/config.py* así como el nombre de los ficheros para no sobreescribirlos (`OUTER_FOLD_FILENAME` y `INNER_FOLD_PREFIX`).

In [7]:
generate_folds(cfg)

Generating outer fold 0

Generating inner fold 0 for outer fold 0
Generating inner fold 1 for outer fold 0
Generating inner fold 2 for outer fold 0
Generating inner fold 3 for outer fold 0
Generating inner fold 4 for outer fold 0
Save complete inner fold information for outer fold 0 in inner_fold_0.csv
------------------------------


Generating outer fold 1

Generating inner fold 0 for outer fold 1
Generating inner fold 1 for outer fold 1
Generating inner fold 2 for outer fold 1
Generating inner fold 3 for outer fold 1
Generating inner fold 4 for outer fold 1
Save complete inner fold information for outer fold 1 in inner_fold_1.csv
------------------------------


Generating outer fold 2

Generating inner fold 0 for outer fold 2
Generating inner fold 1 for outer fold 2
Generating inner fold 2 for outer fold 2
Generating inner fold 3 for outer fold 2
Generating inner fold 4 for outer fold 2
Save complete inner fold information for outer fold 2 in inner_fold_2.csv
----------------------

Este procedimiento garantiza que todos los modelos evaluados se comparen bajo exactamente las mismas particiones de entrenamiento y evaluación. Además, permite reproducir el experimento completo y reutilizar el mismo protocolo para evaluar nuevos algoritmos sobre el conjunto de datos, facilitando la comparación con los resultados obtenidos en este trabajo.

<a id="formato_ficheros_csv"></a>
## 3.1 Formato de los ficheros CSV

La función genera dos tipos de ficheros CSV en el directorio de salida *data/folds*. Estos archivos no contienen directamente las variables del conjunto de datos, sino que actúan como estructuras de mapeo que definen la asignación de cada muestra a los distintos *folds* mediante el identificador único **CC** que contiene el código de complejo residencial.

<a id="outer_folds_csv"></a>
### 3.1.1 Fichero para *Outer CV*: *outer_folds.csv*

Este archivo contiene la asignación de folds para la validación cruzada externa (Outer Cross Validation). Su estructura se compone de las siguientes columnas:
* **CC**. Identificador único del complejo residencial, correspondiente al valor de la variable CC en el conjunto de datos.
* **outer_fold_idx**. Indice del *fold* que indica que dicha muestra pertenece al conjunto de test de ese *fold*. Dado que se utilizan $5$ folds los indices pueden tomar valores en $[0,1,2,3,4]$

A continuación se muestran las cinco primeras filas de este archivo:

In [8]:
outer_df = pd.read_csv(cfg.DATA_FOLDS_DIR/"outer_folds.csv")
outer_df.head()

,CC,outer_fold_idx
0,301015,0
1,301019,0
2,301020,0
3,301012,0
4,301028,0


De este modo, la identificación de los conjuntos de entrenamiento y test para cada fold se realiza a través de la variable *outer_fold_idx*. Por ejemplo, para la primera partición asociada al fold $0$:
* **Conjunto de entrenamiento (*Outer Train Set*)**: todas las muestras cuyo *outer_fold_idx* es distinto de 0.
* **Conjunto de test (*Outer Test Set*)**: todas las muestras cuyo *outer_fold_idx* es igual a 0.


In [9]:
train_outer_df = outer_df[outer_df["outer_fold_idx"] != 0].copy()
test_outer_df = outer_df[outer_df["outer_fold_idx"] == 0].copy()
print("Number of samples in the first fold of Outer Cross Validation (fold 0):")
print(f"Outer Train set: {train_outer_df.shape[0]} samples.")
print(f"Outer Test set: {test_outer_df.shape[0]} samples.")

Number of samples in the first fold of Outer Cross Validation (fold 0):
Outer Train set: 513 samples.
Outer Test set: 129 samples.


<a id="inner_folds_csv"></a>
### 3.1.2 Ficheros para *Inner CV*: *inner_fold_{outer_fold_idx}.csv*

Para cada partición de la validación cruzada externa (*Outer CV*) se genera un fichero independiente que contiene la asignación de folds correspondiente a la validación cruzada interna (*Inner CV*). Dado que la validación externa se compone de $5$ folds, se generan en total cinco ficheros asociados a la validación interna:
* inner_fold_0.csv.
* inner_fold_1.csv.
* inner_fold_2.csv.
* inner_fold_3.csv.
* inner_fold_4.csv.

Cada uno de estos ficheros contiene las siguientes columnas:
* **CC**. Identificador único del complejo residencial que se corresponde con el valor de la variable **CC** en el conjunto de datos.
* **inner_fold_idx**. Indice del *fold* que indica que dicha muestra pertenece al conjunto de validación de ese *fold*. Dado que se utilizan $5$ *folds* los indices pueden tomar valores en $[0,1,2,3,4]$.

A continuación se muestran las cinco primeras filas del fichero *inner_fold_0.csv* asociado al fold $0$ de la validación externa.

In [10]:
inner_df = pd.read_csv(cfg.DATA_FOLDS_DIR/"inner_fold_0.csv")
inner_df.head()

,CC,inner_fold_idx
0,301016,0
1,301027,0
2,2201023,0
3,8709041,0
4,301004,0


Al igual que en el caso anterior, la variable *inner_fold_idx* se utiliza para definir los conjuntos de entrenamiento y validación dentro de cada partición. Por ejemplo, para la primera partición asociada al *Inner fold 0*:
* **Conjunto de entrenamiento (*Inner Train Set*)**: todas las muestras cuyo *inner_fold_idx* es distinto de 0.
* **Conjunto de test (*Inner Validation Set*)**: todas las muestras cuyo *inner_fold_idx* es igual a 0.

A continuación, se muestra el número de muestras correspondientes a los conjuntos de entrenamiento y validación del primer *inner fold*.

In [11]:
train_inner_df = inner_df[inner_df["inner_fold_idx"] != 0].copy()
validation_inner_df = inner_df[inner_df["inner_fold_idx"] == 0].copy()
print("Number of samples in the first fold of Outer Cross Validation (fold 0):")
print(f"Outer Train set: {train_outer_df.shape[0]} samples.")
print(f"Outer Test set: {test_outer_df.shape[0]} samples.")

print("\nNumber of samples in the first fold of the associated Inner Cross Validation:")
print(f"Inner Train Set: {train_inner_df.shape[0]}")
print(f"Inner validation Set: {validation_inner_df.shape[0]}")

Number of samples in the first fold of Outer Cross Validation (fold 0):
Outer Train set: 513 samples.
Outer Test set: 129 samples.

Number of samples in the first fold of the associated Inner Cross Validation:
Inner Train Set: 410
Inner validation Set: 103


Como se puede observar, las $513$ muestras del conjunto de entrenamiento (*Outer Train Set*) de la *Outer CV* se distribuyen en $410$ muestras para el conjunto de entrenamiento (*Inner Train Set*) y $103$ para el conjunto de validación (*Inner Validation Set*) en la primera partición de la *Inner CV*.

<a id="validacion"></a>
# 4. Validación de los folds creados

Para garantizar que los *folds* generados con la función *generate_folds()* son correctos y respetan el esquema *Nested Cross Validation*, se ha desarrollado la función *validate_folds()* (en el módulo *nested_cv_fold.py*).

Para el fichero de la validación cruzada externa (*outer_folds.csv*) se verifica lo siguiente:
* Todas las muestras del *dataset* aparecen en el fichero.
* Cada muestra pertenece a un único *Outer fold* sin duplicidades.

Para cada *fold* de la validación cruzada externa, se verifica que los *inner folds* asociados cumplen:
* Las muestras presentes en el fichero *inner_fold_{outer_fold_idx}.csv*  pertenecen al conjunto de entrenamiento externo (*Outer Train Set*) asociado.
* Cada muestra pertenece a un único *Inner fold* sin duplicidades.

A continuación se muestra la ejecución de dicha función que no detecta ningún error en los ficheros generados.

In [12]:
validate_folds(cfg)


✅ All Nested CV folds are valid.
